# Computational Notebook 05: DeFi Protocols

## Overview

Decentralized Finance (DeFi) reimagines traditional financial services using smart contracts on public blockchains. This notebook builds working simulations of core DeFi primitives: Automated Market Makers (AMMs), lending protocols, flash loans, and yield farming strategies. All implementations use pure Python and numpy to demonstrate the mathematical foundations behind protocols like Uniswap, Aave, and Compound.

## Prerequisites
- Notebook 01: Cryptographic Primitives
- Notebook 03: Ethereum & EVM Analysis
- Notebook 04: Smart Contract Development (ERC-20 tokens)
- Basic understanding of financial concepts (interest rates, liquidity)

## Learning Objectives
1. Implement the constant product AMM formula (x * y = k) and understand price impact
2. Calculate impermanent loss for liquidity providers across different scenarios
3. Build a lending protocol simulator with supply, borrow, and liquidation mechanics
4. Demonstrate flash loan mechanics and arbitrage opportunities
5. Model yield farming strategies and compare APR vs. APY
6. Analyze DeFi protocol risks including oracle, liquidity, and smart contract risk

**Estimated Time:** 4-6 hours

**Related Content:** [Section 03: Ethereum & Smart Contracts](../sections/03-ethereum-smart-contracts.md) | [Section 04: Blockchain Economics](../sections/04-blockchain-economics.md)

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True

print("All imports successful!")
print("This notebook simulates DeFi protocols using Python and numpy.")

---
## 1. Automated Market Makers (AMMs)

Traditional exchanges use order books matching buyers with sellers. AMMs replace this with a mathematical formula that determines prices algorithmically.

### Constant Product Formula

Uniswap V2 uses the **constant product formula**:

$$x \cdot y = k$$

Where:
- $x$ = reserve of token A
- $y$ = reserve of token B  
- $k$ = constant product (invariant)

The spot price is the ratio of reserves: $P = y / x$

In [ ]:
class ConstantProductAMM:
    """Uniswap V2-style constant product AMM."""
    
    FEE_BPS = 30  # 0.30% fee (Uniswap V2)
    
    def __init__(self, token_a: str, token_b: str,
                 reserve_a: float, reserve_b: float) -> None:
        """Initialize pool with reserves."""
        self.token_a = token_a
        self.token_b = token_b
        self.reserve_a = reserve_a
        self.reserve_b = reserve_b
        self.k = reserve_a * reserve_b
        self.lp_total_supply = np.sqrt(reserve_a * reserve_b)
        self.lp_balances: Dict[str, float] = {"initial_lp": self.lp_total_supply}
        self.fee_collected_a = 0.0
        self.fee_collected_b = 0.0
        self.trade_history: List[Dict] = []
    
    def spot_price(self) -> float:
        """Get current spot price of token_a in terms of token_b."""
        return self.reserve_b / self.reserve_a
    
    def get_amount_out(self, amount_in: float, token_in: str) -> Tuple[float, float]:
        """Calculate output amount for a given input (with fee).
        
        Returns (amount_out, price_impact).
        """
        if token_in == self.token_a:
            reserve_in, reserve_out = self.reserve_a, self.reserve_b
        else:
            reserve_in, reserve_out = self.reserve_b, self.reserve_a
        
        # Apply fee
        amount_in_with_fee = amount_in * (10000 - self.FEE_BPS) / 10000
        
        # Constant product: (x + dx) * (y - dy) = x * y
        # dy = y * dx / (x + dx)
        amount_out = reserve_out * amount_in_with_fee / (reserve_in + amount_in_with_fee)
        
        # Price impact
        ideal_out = amount_in * (reserve_out / reserve_in)  # No-impact price
        price_impact = 1 - (amount_out / ideal_out) if ideal_out > 0 else 0
        
        return amount_out, price_impact
    
    def swap(self, amount_in: float, token_in: str, min_out: float = 0) -> float:
        """Execute a swap."""
        amount_out, impact = self.get_amount_out(amount_in, token_in)
        
        if amount_out < min_out:
            raise ValueError(f"Slippage too high: {amount_out:.4f} < {min_out:.4f}")
        
        # Update reserves
        fee = amount_in * self.FEE_BPS / 10000
        if token_in == self.token_a:
            self.reserve_a += amount_in
            self.reserve_b -= amount_out
            self.fee_collected_a += fee
        else:
            self.reserve_b += amount_in
            self.reserve_a -= amount_out
            self.fee_collected_b += fee
        
        self.k = self.reserve_a * self.reserve_b
        
        self.trade_history.append({
            "in": token_in, "amount_in": amount_in,
            "amount_out": amount_out, "impact": impact,
            "price_after": self.spot_price()
        })
        
        return amount_out


# Create an ETH/USDC pool
pool = ConstantProductAMM("ETH", "USDC", 1000.0, 2_000_000.0)

print("=" * 60)
print("CONSTANT PRODUCT AMM (Uniswap V2 Style)")
print("=" * 60)
print(f"Pool: {pool.token_a}/{pool.token_b}")
print(f"Reserves: {pool.reserve_a:,.0f} {pool.token_a} / {pool.reserve_b:,.0f} {pool.token_b}")
print(f"Spot price: 1 {pool.token_a} = ${pool.spot_price():,.2f}")
print(f"k = {pool.k:,.0f}")

# Execute trades of increasing size
print(f"\n{'Trade Size':>12} {'Amount Out':>14} {'Price Impact':>14} {'Effective Price':>16}")
print("-" * 60)

for size in [1, 10, 50, 100, 500]:
    test_pool = ConstantProductAMM("ETH", "USDC", 1000.0, 2_000_000.0)
    out, impact = test_pool.get_amount_out(size, "ETH")
    eff_price = out / size
    print(f"{size:>10} ETH {out:>12,.2f} USDC {impact:>12.4%} ${eff_price:>14,.2f}")

In [ ]:
# Visualize the bonding curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Bonding curve
reserve_a_range = np.linspace(200, 5000, 500)
k = 1000 * 2_000_000  # initial k
reserve_b_range = k / reserve_a_range

axes[0].plot(reserve_a_range, reserve_b_range, 'b-', linewidth=2)
axes[0].plot(1000, 2_000_000, 'ro', markersize=10, label='Current state')
axes[0].set_xlabel('ETH Reserve')
axes[0].set_ylabel('USDC Reserve')
axes[0].set_title('Constant Product Bonding Curve (x * y = k)')
axes[0].legend()
axes[0].set_xlim(0, 5000)
axes[0].set_ylim(0, 10_000_000)

# Right: Price impact vs trade size
trade_sizes = np.linspace(0.1, 500, 200)
impacts = []
for size in trade_sizes:
    _, impact = ConstantProductAMM("ETH", "USDC", 1000, 2_000_000).get_amount_out(size, "ETH")
    impacts.append(impact * 100)

axes[1].plot(trade_sizes, impacts, 'r-', linewidth=2)
axes[1].set_xlabel('Trade Size (ETH)')
axes[1].set_ylabel('Price Impact (%)')
axes[1].set_title('Price Impact vs Trade Size')
axes[1].axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='1% impact')
axes[1].legend()

plt.tight_layout()
plt.savefig('/tmp/amm_bonding_curve.png', dpi=100, bbox_inches='tight')
plt.show()
print("Bonding curve and price impact visualized.")

---
## 2. Liquidity Pools and Impermanent Loss

Liquidity Providers (LPs) deposit pairs of tokens into AMM pools and receive LP tokens representing their share. However, they face **impermanent loss** when token prices diverge from the ratio at deposit time.

### Impermanent Loss Formula

$$IL = \frac{2\sqrt{r}}{1 + r} - 1$$

Where $r$ = price ratio (new price / old price)

In [ ]:
def impermanent_loss(price_ratio: float) -> float:
    """Calculate impermanent loss for a given price change ratio.
    
    Args:
        price_ratio: New price / Original price (e.g., 2.0 means price doubled)
    
    Returns:
        Impermanent loss as a negative fraction (e.g., -0.05 = 5% loss)
    """
    return 2 * np.sqrt(price_ratio) / (1 + price_ratio) - 1


def lp_vs_hold_comparison(initial_a: float, initial_b: float,
                          initial_price: float, new_price: float) -> Dict[str, float]:
    """Compare LP position value vs simple holding.
    
    Args:
        initial_a: Initial amount of token A deposited
        initial_b: Initial amount of token B deposited
        initial_price: Initial price of A in terms of B
        new_price: New price of A in terms of B
    """
    # HODL value
    hold_value = initial_a * new_price + initial_b
    
    # LP position value (after arbitrage to new price)
    # In a constant product AMM: new_a = sqrt(k / new_price), new_b = sqrt(k * new_price)
    k = initial_a * initial_b
    new_a = np.sqrt(k / new_price)
    new_b = np.sqrt(k * new_price)
    lp_value = new_a * new_price + new_b
    
    il = lp_value / hold_value - 1
    
    return {
        "hold_value": hold_value,
        "lp_value": lp_value,
        "il_pct": il * 100,
        "il_absolute": lp_value - hold_value,
        "new_a": new_a,
        "new_b": new_b
    }


# Demonstrate impermanent loss at different price changes
print("=" * 60)
print("IMPERMANENT LOSS ANALYSIS")
print("=" * 60)

print(f"\nDeposit: 10 ETH + 20,000 USDC (ETH price = $2,000)")
print(f"\n{'Price Change':>14} {'New ETH Price':>14} {'HODL Value':>12} {'LP Value':>12} {'IL':>8}")
print("-" * 65)

for change in [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 5.0]:
    new_price = 2000 * change
    result = lp_vs_hold_comparison(10, 20_000, 2000, new_price)
    print(f"{change:>12.2f}x ${new_price:>12,.0f} ${result['hold_value']:>10,.0f} "
          f"${result['lp_value']:>10,.0f} {result['il_pct']:>7.2f}%")

In [ ]:
# Visualize impermanent loss
price_ratios = np.linspace(0.1, 5.0, 500)
il_values = [impermanent_loss(r) * 100 for r in price_ratios]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: IL curve
axes[0].plot(price_ratios, il_values, 'r-', linewidth=2)
axes[0].axhline(y=0, color='black', linewidth=0.5)
axes[0].axvline(x=1, color='gray', linestyle='--', alpha=0.5, label='No price change')
axes[0].set_xlabel('Price Ratio (new / original)')
axes[0].set_ylabel('Impermanent Loss (%)')
axes[0].set_title('Impermanent Loss vs Price Change')

# Annotate key points
for ratio, label in [(0.5, '50% drop'), (2.0, '2x'), (5.0, '5x')]:
    il = impermanent_loss(ratio) * 100
    axes[0].annotate(f'{il:.1f}%', xy=(ratio, il), fontsize=9,
                     xytext=(10, -15), textcoords='offset points')
    axes[0].plot(ratio, il, 'ko', markersize=5)

axes[0].legend()

# Right: Breakeven fee analysis
# How much fees needed to offset IL?
daily_volumes_pct = [0.5, 1.0, 2.0, 5.0]  # Daily volume as % of pool
fee_rate = 0.003  # 0.3%
days = np.arange(1, 366)

for vol_pct in daily_volumes_pct:
    daily_fee_return = vol_pct / 100 * fee_rate
    cumulative_fees = daily_fee_return * days * 100  # as percentage
    axes[1].plot(days, cumulative_fees, label=f'{vol_pct}% daily vol')

# Add IL lines for reference
for ratio, color in [(1.5, 'red'), (2.0, 'darkred')]:
    il = abs(impermanent_loss(ratio)) * 100
    axes[1].axhline(y=il, color=color, linestyle=':', alpha=0.7,
                    label=f'IL at {ratio}x ({il:.1f}%)')

axes[1].set_xlabel('Days')
axes[1].set_ylabel('Cumulative Fee Return (%)')
axes[1].set_title('Fee Returns vs Impermanent Loss')
axes[1].legend(fontsize=8)
axes[1].set_xlim(0, 365)
axes[1].set_ylim(0, 15)

plt.tight_layout()
plt.savefig('/tmp/impermanent_loss.png', dpi=100, bbox_inches='tight')
plt.show()
print("Impermanent loss analysis visualized.")

---
## 3. Lending Protocol Simulation

Lending protocols like Aave and Compound allow users to supply assets to earn interest and borrow against collateral. Key concepts:

- **Collateral Factor**: Maximum borrow as fraction of collateral (e.g., 75%)
- **Liquidation Threshold**: Point at which position can be liquidated (e.g., 80%)
- **Utilization Rate**: Fraction of supplied assets currently borrowed
- **Interest Rate Model**: Dynamic rates based on utilization

In [ ]:
@dataclass
class AssetConfig:
    """Configuration for a lending pool asset."""
    name: str
    price: float                # USD price
    collateral_factor: float    # Max LTV (e.g., 0.75)
    liquidation_threshold: float # Liquidation LTV (e.g., 0.80)
    liquidation_bonus: float    # Bonus for liquidators (e.g., 0.05 = 5%)
    base_rate: float            # Base borrow rate (e.g., 0.02 = 2%)
    slope1: float               # Rate slope below optimal utilization
    slope2: float               # Rate slope above optimal utilization (steep)
    optimal_utilization: float  # Kink point (e.g., 0.80)


@dataclass
class UserPosition:
    """A user's position in the lending protocol."""
    supplied: Dict[str, float] = field(default_factory=lambda: defaultdict(float))
    borrowed: Dict[str, float] = field(default_factory=lambda: defaultdict(float))


class LendingProtocol:
    """Aave/Compound-style lending protocol simulator."""
    
    def __init__(self) -> None:
        """Initialize lending protocol."""
        self.assets: Dict[str, AssetConfig] = {}
        self.total_supplied: Dict[str, float] = defaultdict(float)
        self.total_borrowed: Dict[str, float] = defaultdict(float)
        self.positions: Dict[str, UserPosition] = defaultdict(UserPosition)
        self.events: List[str] = []
    
    def add_asset(self, config: AssetConfig) -> None:
        """Add a supported asset."""
        self.assets[config.name] = config
    
    def get_utilization(self, asset: str) -> float:
        """Calculate utilization rate for an asset."""
        supplied = self.total_supplied[asset]
        if supplied == 0:
            return 0
        return self.total_borrowed[asset] / supplied
    
    def get_borrow_rate(self, asset: str) -> float:
        """Calculate current borrow interest rate."""
        config = self.assets[asset]
        util = self.get_utilization(asset)
        
        if util <= config.optimal_utilization:
            return config.base_rate + (util / config.optimal_utilization) * config.slope1
        else:
            excess = (util - config.optimal_utilization) / (1 - config.optimal_utilization)
            return config.base_rate + config.slope1 + excess * config.slope2
    
    def get_supply_rate(self, asset: str) -> float:
        """Calculate current supply interest rate."""
        util = self.get_utilization(asset)
        borrow_rate = self.get_borrow_rate(asset)
        return borrow_rate * util * 0.9  # 10% reserve factor
    
    def supply(self, user: str, asset: str, amount: float) -> None:
        """Supply assets to the protocol."""
        self.positions[user].supplied[asset] += amount
        self.total_supplied[asset] += amount
        self.events.append(f"SUPPLY: {user[:8]}.. supplied {amount:,.2f} {asset}")
    
    def borrow(self, user: str, asset: str, amount: float) -> None:
        """Borrow assets against collateral."""
        # Check if user has enough collateral
        collateral_value = self._get_collateral_value(user)
        current_borrow_value = self._get_borrow_value(user)
        new_borrow_value = amount * self.assets[asset].price
        
        max_borrow = collateral_value * min(
            cf for a, cf in self._get_weighted_collateral_factors(user)
        ) if collateral_value > 0 else 0
        
        if current_borrow_value + new_borrow_value > collateral_value * 0.75:
            raise ValueError(
                f"Insufficient collateral: need ${current_borrow_value + new_borrow_value:,.0f}, "
                f"max borrow ${collateral_value * 0.75:,.0f}"
            )
        
        if amount > self.total_supplied[asset] - self.total_borrowed[asset]:
            raise ValueError(f"Insufficient liquidity in {asset} pool")
        
        self.positions[user].borrowed[asset] += amount
        self.total_borrowed[asset] += amount
        self.events.append(f"BORROW: {user[:8]}.. borrowed {amount:,.2f} {asset}")
    
    def _get_collateral_value(self, user: str) -> float:
        """Get total collateral value in USD."""
        pos = self.positions[user]
        return sum(amt * self.assets[a].price for a, amt in pos.supplied.items())
    
    def _get_borrow_value(self, user: str) -> float:
        """Get total borrow value in USD."""
        pos = self.positions[user]
        return sum(amt * self.assets[a].price for a, amt in pos.borrowed.items())
    
    def _get_weighted_collateral_factors(self, user: str) -> List[Tuple[str, float]]:
        """Get collateral factors for user's supplied assets."""
        pos = self.positions[user]
        return [(a, self.assets[a].collateral_factor) for a in pos.supplied if pos.supplied[a] > 0]
    
    def get_health_factor(self, user: str) -> float:
        """Calculate health factor (< 1.0 means liquidatable)."""
        pos = self.positions[user]
        collateral_value = sum(
            amt * self.assets[a].price * self.assets[a].liquidation_threshold
            for a, amt in pos.supplied.items()
        )
        borrow_value = self._get_borrow_value(user)
        if borrow_value == 0:
            return float('inf')
        return collateral_value / borrow_value
    
    def check_liquidation(self, user: str) -> Optional[Dict]:
        """Check if a user's position can be liquidated."""
        hf = self.get_health_factor(user)
        if hf >= 1.0:
            return None
        return {
            "user": user,
            "health_factor": hf,
            "collateral_value": self._get_collateral_value(user),
            "borrow_value": self._get_borrow_value(user),
        }
    
    def update_price(self, asset: str, new_price: float) -> None:
        """Update asset price (simulating oracle update)."""
        old_price = self.assets[asset].price
        self.assets[asset].price = new_price
        self.events.append(f"ORACLE: {asset} price ${old_price:,.2f} -> ${new_price:,.2f}")


print("Lending protocol defined.")

In [ ]:
# Set up and demonstrate the lending protocol
protocol = LendingProtocol()

# Add assets
protocol.add_asset(AssetConfig(
    "ETH", 2000, 0.80, 0.825, 0.05, 0.02, 0.04, 0.75, 0.80
))
protocol.add_asset(AssetConfig(
    "USDC", 1.0, 0.85, 0.90, 0.05, 0.01, 0.04, 0.60, 0.90
))
protocol.add_asset(AssetConfig(
    "WBTC", 40000, 0.70, 0.75, 0.10, 0.02, 0.04, 0.80, 0.80
))

# Simulate users
print("=" * 60)
print("LENDING PROTOCOL SIMULATION")
print("=" * 60)

# Suppliers provide liquidity
protocol.supply("supplier_1", "ETH", 100)
protocol.supply("supplier_2", "USDC", 500_000)
protocol.supply("supplier_3", "WBTC", 5)

# Borrower deposits ETH, borrows USDC
protocol.supply("borrower_1", "ETH", 10)  # $20,000 collateral
protocol.borrow("borrower_1", "USDC", 12_000)  # Borrow $12,000 (60% LTV)

print(f"\n--- Pool Status ---")
for asset in ["ETH", "USDC", "WBTC"]:
    util = protocol.get_utilization(asset)
    borrow_rate = protocol.get_borrow_rate(asset)
    supply_rate = protocol.get_supply_rate(asset)
    print(f"  {asset:>4}: Supplied={protocol.total_supplied[asset]:>10,.2f}  "
          f"Borrowed={protocol.total_borrowed[asset]:>10,.2f}  "
          f"Util={util:>6.1%}  Borrow={borrow_rate:>6.2%}  Supply={supply_rate:>6.2%}")

# Check borrower health
hf = protocol.get_health_factor("borrower_1")
print(f"\n--- Borrower 1 Position ---")
print(f"  Collateral: 10 ETH = ${protocol._get_collateral_value('borrower_1'):,.0f}")
print(f"  Borrowed: 12,000 USDC = ${protocol._get_borrow_value('borrower_1'):,.0f}")
print(f"  Health Factor: {hf:.2f} (safe > 1.0)")

In [ ]:
# Simulate price crash and liquidation
print("=" * 60)
print("LIQUIDATION SCENARIO")
print("=" * 60)

print(f"\nSimulating ETH price crash...")
prices = [2000, 1800, 1600, 1500, 1400, 1300, 1200]

print(f"\n{'ETH Price':>10} {'Collateral':>12} {'Borrowed':>10} {'Health':>8} {'Status':>12}")
print("-" * 55)

for price in prices:
    protocol.update_price("ETH", price)
    collateral = protocol._get_collateral_value("borrower_1")
    borrowed = protocol._get_borrow_value("borrower_1")
    hf = protocol.get_health_factor("borrower_1")
    liq = protocol.check_liquidation("borrower_1")
    status = "LIQUIDATABLE" if liq else "Safe"
    print(f"${price:>9,} ${collateral:>10,.0f} ${borrowed:>8,.0f} {hf:>7.2f}  {status:>12}")

print(f"\nAt ETH=$1,400 the position becomes liquidatable!")
print(f"Liquidator can repay some of the debt and claim collateral + 5% bonus.")

In [ ]:
# Visualize interest rate model
fig, ax = plt.subplots(figsize=(10, 6))

utilizations = np.linspace(0, 1, 1000)

for asset_name, color in [("ETH", "blue"), ("USDC", "green"), ("WBTC", "orange")]:
    config = protocol.assets[asset_name]
    rates = []
    for u in utilizations:
        if u <= config.optimal_utilization:
            rate = config.base_rate + (u / config.optimal_utilization) * config.slope1
        else:
            excess = (u - config.optimal_utilization) / (1 - config.optimal_utilization)
            rate = config.base_rate + config.slope1 + excess * config.slope2
        rates.append(rate * 100)
    
    ax.plot(utilizations * 100, rates, color=color, linewidth=2, label=asset_name)
    ax.axvline(x=config.optimal_utilization * 100, color=color, linestyle=':', alpha=0.3)

ax.set_xlabel('Utilization Rate (%)')
ax.set_ylabel('Borrow Interest Rate (%)')
ax.set_title('Interest Rate Model (Kink-Style)')
ax.legend()
ax.set_xlim(0, 100)

plt.tight_layout()
plt.savefig('/tmp/interest_rate_model.png', dpi=100, bbox_inches='tight')
plt.show()
print("Interest rate model visualized.")
print("Note the steep increase above the optimal utilization (kink point).")

---
## 4. Flash Loans

Flash loans are uncollateralized loans that must be borrowed and repaid within a single transaction. If the borrower can't repay, the entire transaction reverts as if it never happened.

### Key Properties
- No collateral required
- Must repay principal + fee in same transaction
- Atomic: all-or-nothing execution
- Primary use: arbitrage between DEXs

In [ ]:
class FlashLoanProvider:
    """Simulate flash loan mechanics."""
    
    FEE_BPS = 9  # 0.09% (Aave V3 fee)
    
    def __init__(self, reserves: Dict[str, float]) -> None:
        """Initialize with available reserves."""
        self.reserves = dict(reserves)
        self.fees_collected: Dict[str, float] = defaultdict(float)
    
    def flash_loan(self, asset: str, amount: float,
                   callback: callable) -> Dict[str, float]:
        """Execute a flash loan.
        
        Args:
            asset: Token to borrow
            amount: Amount to borrow
            callback: Function that receives the borrowed amount and must return repayment
        
        Returns:
            Result dict with profit/loss info
        """
        if amount > self.reserves[asset]:
            raise ValueError(f"Insufficient reserves: {self.reserves[asset]} < {amount}")
        
        fee = amount * self.FEE_BPS / 10000
        required_repayment = amount + fee
        
        # Lend out the funds
        self.reserves[asset] -= amount
        
        # Execute the callback (user's arbitrage logic)
        try:
            repayment = callback(asset, amount)
            
            if repayment < required_repayment:
                # Transaction reverts!
                self.reserves[asset] += amount  # Undo the loan
                raise ValueError(
                    f"Flash loan not repaid: returned {repayment:.2f}, "
                    f"needed {required_repayment:.2f}"
                )
            
            # Repayment successful
            self.reserves[asset] += required_repayment
            self.fees_collected[asset] += fee
            
            profit = repayment - required_repayment
            return {
                "success": True,
                "borrowed": amount,
                "fee": fee,
                "repaid": required_repayment,
                "profit": profit
            }
        except Exception as e:
            self.reserves[asset] += amount  # Revert
            raise


# Simulate flash loan arbitrage between two DEXs
print("=" * 60)
print("FLASH LOAN ARBITRAGE SIMULATION")
print("=" * 60)

# Two DEXs with different ETH prices
dex_a = ConstantProductAMM("ETH", "USDC", 500, 1_000_000)  # ETH = $2,000
dex_b = ConstantProductAMM("ETH", "USDC", 500, 1_050_000)  # ETH = $2,100

print(f"DEX A: ETH = ${dex_a.spot_price():,.2f}")
print(f"DEX B: ETH = ${dex_b.spot_price():,.2f}")
print(f"Price gap: ${dex_b.spot_price() - dex_a.spot_price():,.2f} ({(dex_b.spot_price()/dex_a.spot_price()-1)*100:.2f}%)")

# Flash loan provider
provider = FlashLoanProvider({"USDC": 10_000_000, "ETH": 5000})

def arbitrage_callback(asset: str, amount: float) -> float:
    """Buy ETH cheap on DEX A, sell expensive on DEX B."""
    # Step 1: Buy ETH on DEX A (cheaper)
    eth_received = dex_a.swap(amount, "USDC")
    
    # Step 2: Sell ETH on DEX B (more expensive)
    usdc_received = dex_b.swap(eth_received, "ETH")
    
    return usdc_received

# Execute flash loan
loan_amount = 10_000  # Borrow 10,000 USDC
result = provider.flash_loan("USDC", loan_amount, arbitrage_callback)

print(f"\n--- Flash Loan Execution ---")
print(f"Borrowed:  {result['borrowed']:>12,.2f} USDC")
print(f"Fee:       {result['fee']:>12,.2f} USDC")
print(f"Repaid:    {result['repaid']:>12,.2f} USDC")
print(f"Profit:    {result['profit']:>12,.2f} USDC")
print(f"\nAfter arbitrage:")
print(f"DEX A: ETH = ${dex_a.spot_price():,.2f} (increased - we bought)")
print(f"DEX B: ETH = ${dex_b.spot_price():,.2f} (decreased - we sold)")
print(f"Prices converged! Arbitrage corrects market inefficiencies.")

---
## 5. Yield Farming Strategies

Yield farming involves deploying crypto assets across DeFi protocols to maximize returns. Key metrics:

- **APR** (Annual Percentage Rate): Simple interest, not compounded
- **APY** (Annual Percentage Yield): Compound interest

$$APY = \left(1 + \frac{APR}{n}\right)^n - 1$$

where $n$ = number of compounding periods per year

In [ ]:
def apr_to_apy(apr: float, compounds_per_year: int) -> float:
    """Convert APR to APY given compounding frequency."""
    return (1 + apr / compounds_per_year) ** compounds_per_year - 1


@dataclass
class FarmPosition:
    """A yield farming position."""
    protocol: str
    principal: float
    apr: float
    compounds_per_year: int
    reward_token: str = "REWARD"
    reward_apr: float = 0.0  # Additional reward token APR


class YieldFarmSimulator:
    """Simulate yield farming strategies over time."""
    
    def __init__(self, positions: List[FarmPosition]) -> None:
        """Initialize with farming positions."""
        self.positions = positions
    
    def simulate(self, days: int) -> Dict[str, List[float]]:
        """Simulate positions over time.
        
        Returns dict of position label -> daily values.
        """
        results = {}
        for pos in self.positions:
            daily_rate = pos.apr / 365
            compound_interval = 365 / pos.compounds_per_year
            
            values = [pos.principal]
            current = pos.principal
            accrued_interest = 0
            
            for day in range(1, days + 1):
                accrued_interest += current * daily_rate
                
                # Compound at intervals
                if day % max(1, int(compound_interval)) == 0:
                    current += accrued_interest
                    accrued_interest = 0
                
                values.append(current + accrued_interest)
            
            label = f"{pos.protocol} ({pos.apr*100:.0f}% APR, {pos.compounds_per_year}x/yr)"
            results[label] = values
        
        return results


# Compare APR vs APY
print("=" * 60)
print("APR vs APY COMPARISON")
print("=" * 60)

apr = 0.20  # 20% APR
print(f"\nBase APR: {apr*100:.0f}%")
print(f"\n{'Compounding':>20} {'APY':>10} {'$10,000 after 1yr':>20}")
print("-" * 55)

for label, n in [("Annual", 1), ("Quarterly", 4), ("Monthly", 12),
                  ("Weekly", 52), ("Daily", 365), ("Hourly", 8760)]:
    apy = apr_to_apy(apr, n)
    final = 10_000 * (1 + apy)
    print(f"{label:>20} {apy*100:>9.2f}% ${final:>18,.2f}")

In [ ]:
# Simulate different yield farming strategies
positions = [
    FarmPosition("Aave USDC", 10_000, 0.05, 365),       # 5% APR, daily compound
    FarmPosition("Uniswap LP", 10_000, 0.25, 1),        # 25% APR, annual (no auto-compound)
    FarmPosition("Compound ETH", 10_000, 0.08, 365),    # 8% APR, daily compound
    FarmPosition("Yearn Vault", 10_000, 0.15, 365),     # 15% APR, daily compound (auto)
]

simulator = YieldFarmSimulator(positions)
results = simulator.simulate(365)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

days = np.arange(0, 366)
colors = ['blue', 'red', 'green', 'purple']

for (label, values), color in zip(results.items(), colors):
    axes[0].plot(days, values, color=color, linewidth=2, label=label)

axes[0].set_xlabel('Days')
axes[0].set_ylabel('Position Value ($)')
axes[0].set_title('Yield Farming Strategy Comparison (1 Year)')
axes[0].legend(fontsize=8)

# Final values bar chart
labels = [p.protocol for p in positions]
final_values = [results[k][-1] for k in results]
profits = [v - 10_000 for v in final_values]

bars = axes[1].bar(labels, profits, color=colors, alpha=0.7)
axes[1].set_ylabel('Profit ($)')
axes[1].set_title('Total Profit After 1 Year ($10,000 Initial)')
for bar, profit in zip(bars, profits):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'${profit:,.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/yield_farming.png', dpi=100, bbox_inches='tight')
plt.show()
print("Yield farming strategy comparison visualized.")

---
## 6. DEX Aggregation

DEX aggregators like 1inch route trades across multiple pools to find the best execution price. This is particularly important for large trades where a single pool would cause significant price impact.

In [ ]:
class DEXAggregator:
    """Simulate DEX aggregation across multiple pools."""
    
    def __init__(self, pools: List[ConstantProductAMM]) -> None:
        """Initialize with available pools."""
        self.pools = pools
    
    def find_best_single_pool(self, amount_in: float,
                               token_in: str) -> Tuple[int, float, float]:
        """Find the best single pool for a trade.
        
        Returns (pool_index, amount_out, price_impact).
        """
        best_out = 0
        best_idx = 0
        best_impact = 0
        
        for i, pool in enumerate(self.pools):
            out, impact = pool.get_amount_out(amount_in, token_in)
            if out > best_out:
                best_out = out
                best_idx = i
                best_impact = impact
        
        return best_idx, best_out, best_impact
    
    def find_optimal_split(self, amount_in: float, token_in: str,
                           n_splits: int = 10) -> Tuple[List[float], float, float]:
        """Find optimal split across pools to minimize price impact.
        
        Returns (splits, total_out, avg_impact).
        """
        n_pools = len(self.pools)
        best_total = 0
        best_splits = [0] * n_pools
        
        # Simple grid search over split ratios
        step = 1.0 / n_splits
        
        if n_pools == 2:
            for i in range(n_splits + 1):
                ratio = i * step
                splits = [amount_in * ratio, amount_in * (1 - ratio)]
                total_out = 0
                for j, pool in enumerate(self.pools):
                    if splits[j] > 0:
                        out, _ = pool.get_amount_out(splits[j], token_in)
                        total_out += out
                if total_out > best_total:
                    best_total = total_out
                    best_splits = list(splits)
        elif n_pools == 3:
            for i in range(n_splits + 1):
                for j in range(n_splits + 1 - i):
                    r1 = i * step
                    r2 = j * step
                    r3 = 1 - r1 - r2
                    splits = [amount_in * r1, amount_in * r2, amount_in * r3]
                    total_out = 0
                    for k, pool in enumerate(self.pools):
                        if splits[k] > 0:
                            out, _ = pool.get_amount_out(splits[k], token_in)
                            total_out += out
                    if total_out > best_total:
                        best_total = total_out
                        best_splits = list(splits)
        
        ideal = amount_in * max(p.spot_price() for p in self.pools)
        avg_impact = 1 - (best_total / ideal) if ideal > 0 else 0
        
        return best_splits, best_total, avg_impact


# Demonstrate DEX aggregation
print("=" * 60)
print("DEX AGGREGATION - SPLIT ROUTING")
print("=" * 60)

# Three pools with different depths
pools = [
    ConstantProductAMM("ETH", "USDC", 5000, 10_000_000),   # Deep pool
    ConstantProductAMM("ETH", "USDC", 1000, 2_000_000),    # Medium pool
    ConstantProductAMM("ETH", "USDC", 200, 400_000),       # Shallow pool
]

print(f"\nPools:")
for i, p in enumerate(pools):
    print(f"  Pool {i}: {p.reserve_a:,.0f} ETH / {p.reserve_b:,.0f} USDC (price: ${p.spot_price():,.2f})")

aggregator = DEXAggregator(pools)

# Compare single pool vs split for large trade
trade_size = 200  # 200 ETH - large trade

print(f"\n--- Trading {trade_size} ETH ---")

# Single pool (best)
idx, single_out, single_impact = aggregator.find_best_single_pool(trade_size, "ETH")
print(f"\nBest single pool (Pool {idx}):")
print(f"  Output: {single_out:,.2f} USDC")
print(f"  Price impact: {single_impact:.4%}")
print(f"  Effective price: ${single_out/trade_size:,.2f}")

# Split across pools
splits, split_out, split_impact = aggregator.find_optimal_split(trade_size, "ETH", 20)
print(f"\nOptimal split:")
for i, s in enumerate(splits):
    if s > 0:
        print(f"  Pool {i}: {s:,.1f} ETH ({s/trade_size*100:.0f}%)")
print(f"  Output: {split_out:,.2f} USDC")
print(f"  Avg price impact: {split_impact:.4%}")
print(f"  Effective price: ${split_out/trade_size:,.2f}")

improvement = split_out - single_out
print(f"\nSplit routing saves: {improvement:,.2f} USDC ({improvement/single_out*100:.2f}%)")

---
## 7. DeFi Risk Analysis

DeFi protocols face multiple risk categories. Understanding these risks is essential for participants and builders.

In [ ]:
@dataclass
class ProtocolRisk:
    """Risk assessment for a DeFi protocol."""
    name: str
    smart_contract_risk: float    # 0-10 (0 = no risk, 10 = maximum)
    oracle_risk: float            # Dependency on price oracles
    liquidity_risk: float         # Risk of liquidity drying up
    governance_risk: float        # Centralization of control
    economic_risk: float          # Tokenomics / incentive failures
    audit_score: float            # Quality of audits (higher = better = lower risk)
    tvl_billions: float           # Total Value Locked
    age_months: int               # Time since launch


# Analyze several protocols
protocols = [
    ProtocolRisk("Uniswap V3", 2, 1, 2, 2, 2, 9, 5.0, 36),
    ProtocolRisk("Aave V3", 2, 4, 2, 3, 2, 9, 10.0, 30),
    ProtocolRisk("Curve", 3, 3, 2, 4, 3, 8, 4.0, 36),
    ProtocolRisk("Compound", 2, 4, 3, 3, 2, 9, 3.0, 48),
    ProtocolRisk("New DeFi X", 7, 6, 7, 8, 7, 3, 0.1, 2),
]

print("=" * 60)
print("DEFI PROTOCOL RISK ANALYSIS")
print("=" * 60)

def composite_risk(p: ProtocolRisk) -> float:
    """Calculate composite risk score (0-10)."""
    # Weighted average of risk factors
    raw = (p.smart_contract_risk * 0.25 +
           p.oracle_risk * 0.15 +
           p.liquidity_risk * 0.15 +
           p.governance_risk * 0.15 +
           p.economic_risk * 0.15 +
           (10 - p.audit_score) * 0.15)  # Invert audit score
    
    # Adjust for maturity (older = lower risk)
    maturity_factor = max(0.7, 1 - p.age_months / 120)
    return raw * maturity_factor

print(f"\n{'Protocol':<15} {'SC':>4} {'Oracle':>7} {'Liq':>5} {'Gov':>5} {'Econ':>5} {'Audit':>6} {'Composite':>10}")
print("-" * 65)
for p in protocols:
    score = composite_risk(p)
    risk_level = "LOW" if score < 3 else "MEDIUM" if score < 5 else "HIGH"
    print(f"{p.name:<15} {p.smart_contract_risk:>4.0f} {p.oracle_risk:>7.0f} "
          f"{p.liquidity_risk:>5.0f} {p.governance_risk:>5.0f} {p.economic_risk:>5.0f} "
          f"{p.audit_score:>6.0f} {score:>7.1f} ({risk_level})")

In [ ]:
# Radar chart comparison
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

categories = ['Smart Contract', 'Oracle', 'Liquidity', 'Governance', 'Economic', 'Audit Gap']
n_cats = len(categories)
angles = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

colors = ['blue', 'green', 'orange', 'purple', 'red']
for p, color in zip(protocols, colors):
    values = [p.smart_contract_risk, p.oracle_risk, p.liquidity_risk,
              p.governance_risk, p.economic_risk, 10 - p.audit_score]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=p.name, alpha=0.7)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_ylim(0, 10)
ax.set_title('DeFi Protocol Risk Comparison', size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig('/tmp/defi_risk_radar.png', dpi=100, bbox_inches='tight')
plt.show()
print("Risk comparison radar chart generated.")
print("Smaller area = lower risk. Established protocols cluster near the center.")

---
## Exercises

### Exercise 1: Concentrated Liquidity (Uniswap V3 Style)

Implement a concentrated liquidity AMM where LPs provide liquidity within a specific price range instead of across all prices. Calculate capital efficiency improvement over V2.

**Hints:**
- In V3, liquidity is only active within the LP's chosen range [P_low, P_high]
- Capital efficiency = V2 liquidity needed / V3 liquidity needed for same depth
- Think of it as a virtual reserve curve shifted to focus on a price range

In [ ]:
class ConcentratedLiquidityAMM:
    """Uniswap V3-style concentrated liquidity AMM."""
    
    def __init__(self, token_a: str, token_b: str,
                 current_price: float) -> None:
        """Initialize with current price."""
        self.token_a = token_a
        self.token_b = token_b
        self.current_price = current_price
        # YOUR CODE HERE - add position tracking
    
    def add_liquidity(self, provider: str, amount_a: float, amount_b: float,
                      price_low: float, price_high: float) -> None:
        """Add concentrated liquidity within a price range."""
        # YOUR CODE HERE
        pass
    
    def calculate_capital_efficiency(self, price_low: float,
                                     price_high: float) -> float:
        """Calculate capital efficiency vs V2 full-range."""
        # YOUR CODE HERE
        pass

### Exercise 2: Liquidation Bot Simulator

Build a liquidation bot that monitors lending positions and executes profitable liquidations.

**Hints:**
- Monitor health factors of all borrowers
- Calculate profit from liquidation bonus minus gas costs
- Use flash loans to fund liquidations without capital

In [ ]:
class LiquidationBot:
    """Automated liquidation bot for lending protocols."""
    
    def __init__(self, protocol: LendingProtocol, gas_cost_usd: float = 50) -> None:
        """Initialize bot with protocol reference and gas cost estimate."""
        self.protocol = protocol
        self.gas_cost = gas_cost_usd
        # YOUR CODE HERE
    
    def scan_positions(self) -> List[Dict]:
        """Scan all positions for liquidation opportunities."""
        # YOUR CODE HERE
        pass
    
    def execute_liquidation(self, user: str) -> Dict:
        """Execute a liquidation if profitable."""
        # YOUR CODE HERE
        pass

### Exercise 3: Yield Optimizer

Build a yield optimizer that automatically compounds rewards across multiple farms and rebalances to the highest-yielding strategies.

**Hints:**
- Track multiple farming positions simultaneously
- Factor in gas costs for harvesting and rebalancing
- Consider minimum harvest thresholds to avoid wasteful gas spending

In [ ]:
class YieldOptimizer:
    """Auto-compounding yield optimizer."""
    
    def __init__(self, initial_capital: float, gas_cost_usd: float = 10) -> None:
        """Initialize optimizer with capital and gas costs."""
        self.capital = initial_capital
        self.gas_cost = gas_cost_usd
        # YOUR CODE HERE
    
    def add_farm(self, name: str, apr: float, min_deposit: float = 0) -> None:
        """Add a farming opportunity."""
        # YOUR CODE HERE
        pass
    
    def optimize(self, days: int) -> Dict:
        """Run optimization over given period."""
        # YOUR CODE HERE
        pass

---
## Summary

### What You Learned
- [x] Constant product AMM formula (x * y = k) and price impact mechanics
- [x] Impermanent loss calculation and when it matters for LPs
- [x] Lending protocol mechanics: collateral, borrowing, interest rates, liquidation
- [x] Flash loan mechanics and how they enable zero-capital arbitrage
- [x] Yield farming strategies and APR vs APY calculations
- [x] DEX aggregation and optimal trade routing
- [x] DeFi risk assessment framework

### Key Takeaways
1. **AMMs trade simplicity for capital efficiency** - concentrated liquidity (V3) improves this
2. **Impermanent loss is the cost of providing liquidity** - fees must exceed IL for profit
3. **Flash loans are a DeFi primitive** - they enable capital-efficient arbitrage
4. **Higher yields = higher risks** - always assess protocol risks before depositing
5. **Composability is DeFi's superpower and risk** - protocols build on each other

### Next Steps
- [Notebook 06: Market Analysis](06-market-analysis.ipynb) - Technical analysis and market microstructure
- [Section 04: Blockchain Economics](../sections/04-blockchain-economics.md) - Economic theory behind DeFi